# Ожидания к датасету

Записали пороги до того, как смотреть в данные, чтобы не подгонять их
под результат. Часть А это проверки и часть Б это гипотезы

## Часть А. Проверки

| что проверяем | порог | комментарий |
|---|---|---|
| колонки и типы | все на месте | |
| `rating` | только 1–5 | |
| `timestamp` vs партиция `year` | совпадает | |
| `helpful_vote` | ≥ 0 | |
| пропуски в `user_id`, `parent_asin`, `rating`, `timestamp` | 0% |  |
| полные дубли строк | 0 | |
| повторные отзывы юзера на один товар | < 1% |  |
| `parent_asin` в meta | уникален |  |
| отзывы, у которых товар есть в meta | ≥ 99% | |
| боты | нет | |
| пустые / короткие (< 3 слов) тексты | < 5% |  |

## Часть Б. Гипотезы

Считаем на данных после чистки по части А.

| что проверяем | ожидаем | что это даст |
|---|---|---|
| разреженность матрицы | > 99.9% | |
| юзеры с одним отзывом | > 50% |  |
| интеракции по времени | без провалов |  |
| `rating` ≥ 4 | ≥ 70% отзывов | |
| `title` у айтемов | ≥ 95% | |
| непустой `description` | ≥ 70% | |
| картинки | ≥ 90% | |
| контент у холодных vs тёплых | разница ≤ 10 п.п. |  |



## Подключение к S3

In [ ]:
import os

import pandas as pd
import truststore

truststore.inject_into_ssl()

"AWS_ACCESS_KEY_ID" in os.environ, "AWS_SECRET_ACCESS_KEY" in os.environ

(True, True)

In [16]:
import s3fs

fs = s3fs.S3FileSystem(
    endpoint_url="https://s3.ru-7.storage.selcloud.ru",
    client_kwargs={"region_name": "ru-7"},
)
fs.ls("hse-recsys-project")

['hse-recsys-project/data']

In [17]:
fs.ls("hse-recsys-project/data/processed")

['hse-recsys-project/data/processed/meta',
 'hse-recsys-project/data/processed/reviews']

In [18]:
fs.ls("hse-recsys-project/data/processed/reviews")

['hse-recsys-project/data/processed/reviews/amazon_fashion',
 'hse-recsys-project/data/processed/reviews/grocery_and_gourmet_food']

In [19]:
fs.ls("hse-recsys-project/data/processed/reviews/amazon_fashion")

['hse-recsys-project/data/processed/reviews/amazon_fashion/_SUCCESS',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2002',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2003',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2004',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2005',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2006',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2007',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2008',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2009',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2010',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2011',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2012',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2013',
 'hse-recsys-project/data/processed/reviews/amazon_fashion/year=2014',
 'hse-r

отзывы разложены по годам с 2002 по 2023

### Схема

In [20]:
import pyarrow.dataset as ds

base = "hse-recsys-project/data/processed"
for kind in ["reviews", "meta"]:
    d = ds.dataset(f"{base}/{kind}/amazon_fashion", filesystem=fs, partitioning="hive")
    print(kind)
    print(d.schema)
    print()

reviews
rating: int8
title: large_string
text: large_string
asin: large_string
parent_asin: large_string
user_id: large_string
timestamp: timestamp[ms]
helpful_vote: int32
verified_purchase: bool
n_images: int16
year: int32
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1337

meta
parent_asin: string
main_category: string
title: string
store: string
price: double
average_rating: double
rating_number: int64
features: list<element: string>
  child 0, element: string
description: list<element: string>
  child 0, element: string
categories: list<element: string>
  child 0, element: string
images: string
videos: string
details: string
bought_together: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1759



In [21]:
meta_ds = ds.dataset(f"{base}/meta/amazon_fashion", filesystem=fs)
for field in meta_ds.schema:
    print(field.name, field.type)

parent_asin string
main_category string
title string
store string
price double
average_rating double
rating_number int64
features list<element: string>
description list<element: string>
categories list<element: string>
images string
videos string
details string
bought_together string


In [27]:
sample = meta_ds.head(3).to_pandas()
sample

,parent_asin,main_category,title,store,price,average_rating,rating_number,features,description,categories,images,videos,details,bought_together
0,B08BHN9PK5,AMAZON FASHION,YUEDGE 5 Pairs Men's Moisture Control Cushione...,GiveGift,NaN,4.6,16,[],[],[],"[{""thumb"": ""https://m.media-amazon.com/images/...",[],"{""Package Dimensions"": ""10.31 x 8.5 x 1.73 inc...",NaN
1,B08R39MRDW,AMAZON FASHION,DouBCQ Women's Palazzo Lounge Wide Leg Casual ...,DouBCQ,NaN,4.1,7,"[Drawstring closure, Machine Wash]",[],[],"[{""thumb"": ""https://m.media-amazon.com/images/...",[],"{""Package Dimensions"": ""15 x 10.2 x 0.4 inches...",NaN
2,B077KJHCJ4,AMAZON FASHION,Pastel by Vivienne Honey Vanilla Girls' Trapez...,Pastel by Vivienne,NaN,4.3,11,"[Zipper closure, Hand Wash Only]",[],[],"[{""thumb"": ""https://m.media-amazon.com/images/...",[],"{""Is Discontinued By Manufacturer"": ""No"", ""Pac...",NaN


In [28]:
print(sample.loc[0, "images"][:500])

[{"thumb": "https://m.media-amazon.com/images/I/41+cCfaVOFS._AC_SR38,50_.jpg", "large": "https://m.media-amazon.com/images/I/41+cCfaVOFS._AC_.jpg", "variant": "MAIN", "hi_res": "https://m.media-amazon.com/images/I/81XlFXImFrS._AC_UL1500_.jpg"}, {"thumb": "https://m.media-amazon.com/images/I/41jBdP7etRS._AC_SR38,50_.jpg", "large": "https://m.media-amazon.com/images/I/41jBdP7etRS._AC_.jpg", "variant": "PT01", "hi_res": "https://m.media-amazon.com/images/I/61+yVkHHQ3S._AC_UL1200_.jpg"}, {"thumb": "


images хранится как JSON-строка со списком фото. на первых товарах пустые description, categories, price, проверим долю на всех данных

### Загрузка отзывов

In [29]:
reviews_ds = ds.dataset(
    f"{base}/reviews/amazon_fashion", filesystem=fs, partitioning="hive"
)

cols = [
    "user_id",
    "parent_asin",
    "asin",
    "rating",
    "timestamp",
    "helpful_vote",
    "verified_purchase",
    "n_images",
    "year",
]

reviews = reviews_ds.to_table(columns=cols).to_pandas()
reviews.shape

(2500939, 9)

In [30]:
reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 2500939 entries, 0 to 2500938
Data columns (total 9 columns):
 #   Column             Dtype         
---  ------             -----         
 0   user_id            str           
 1   parent_asin        str           
 2   asin               str           
 3   rating             int8          
 4   timestamp          datetime64[ms]
 5   helpful_vote       int32         
 6   verified_purchase  bool          
 7   n_images           int16         
 8   year               int32         
dtypes: bool(1), datetime64[ms](1), int16(1), int32(2), int8(1), str(3)
memory usage: 219.4 MB


## Часть А. проверки 

### Пропуски

In [32]:
reviews.isna().sum()

user_id              0
parent_asin          0
asin                 0
rating               0
timestamp            0
helpful_vote         0
verified_purchase    0
n_images             0
year                 0
dtype: int64

пропусков нет ни в одной колонке

### Значения

In [33]:
results = []


def expect(name, fact, ok, comment=""):
    results.append({"проверка": name, "факт": fact, "ok": ok, "комментарий": comment})

In [34]:
key_cols = ["user_id", "parent_asin", "rating", "timestamp"]
n_missing = int(reviews[key_cols].isna().sum().sum())
n_empty_ids = int(
    (reviews["user_id"] == "").sum() + (reviews["parent_asin"] == "").sum()
)
expect(
    "пропуски в ключевых колонках",
    n_missing + n_empty_ids,
    n_missing + n_empty_ids == 0,
)

bad_rating = int((~reviews["rating"].between(1, 5)).sum())
expect("rating только 1–5", bad_rating, bad_rating == 0)

year_mismatch = int((reviews["timestamp"].dt.year != reviews["year"]).sum())
expect("timestamp совпадает с year", year_mismatch, year_mismatch == 0)

neg_votes = int((reviews["helpful_vote"] < 0).sum())
expect("helpful_vote ≥ 0", neg_votes, neg_votes == 0)

pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,


схема и значения корректны: оценки 1-5, даты совпадают с партициями, отрицательных helpful_vote нет 

### Дубликаты 

In [35]:
n_full_dup = int(reviews.duplicated().sum())
expect("полные дубли строк", n_full_dup, n_full_dup == 0, "сравнивали без title и text")

repeat_share = reviews.duplicated(["user_id", "parent_asin"]).mean()
expect("повторные отзывы юзера на товар", f"{repeat_share:.2%}", repeat_share < 0.01)

pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,


In [37]:
dedup = reviews.drop_duplicates()
real_repeat_share = dedup.duplicated(["user_id", "parent_asin"]).mean()
print(len(dedup), f"{real_repeat_share:.2%}")

2475695 0.05%


In [38]:
expect(
    "повторы после удаления дублей",
    f"{real_repeat_share:.2%}",
    real_repeat_share < 0.01,
    "оставляем, могут быть разные размеры/цвета",
)
reviews_clean = dedup
pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,
6,повторы после удаления дублей,0.05%,True,"оставляем, могут быть разные размеры/цвета"


дубликаты: нашли 25 244 полных дубля: совпадают юзер,
товар, оценка и время до миллисекунды. это артефакт данных, в ETL удаляем.

повторные отзывы юзера на один товар 1.06%, но почти все они
и есть эти дубли. после удаления дублей остаётся 0.05% честных повторов,
порог < 1% выполняется. они могут быть покупками разных размеров или
цветов одного товара (`parent_asin` их объединяет), оставляем

### Связь с каталогом 

In [39]:
meta_ids = meta_ds.to_table(columns=["parent_asin"]).to_pandas()

n_meta_dup = int(meta_ids["parent_asin"].duplicated().sum())
expect(
    "parent_asin в meta уникален",
    n_meta_dup,
    n_meta_dup == 0,
    "каждый товар один раз в каталоге",
)

in_meta_share = reviews_clean["parent_asin"].isin(meta_ids["parent_asin"]).mean()
expect("товар отзыва есть в meta", f"{in_meta_share:.2%}", in_meta_share >= 0.99)

pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,
6,повторы после удаления дублей,0.05%,True,"оставляем, могут быть разные размеры/цвета"
7,parent_asin в meta уникален,0,True,каждый товар один раз в каталоге
8,товар отзыва есть в meta,100.00%,True,


каждый товар встречается в каталоге один раз. карточка товара есть практически у всех отзывов. склеивать отзывы с каталогом безопасно

### Боты 

In [40]:
per_user = reviews_clean.groupby("user_id").size()
per_user.describe(percentiles=[0.5, 0.9, 0.99, 0.999])

count    2.035490e+06
mean     1.216265e+00
std      9.113782e-01
min      1.000000e+00
50%      1.000000e+00
90%      2.000000e+00
99%      4.000000e+00
99.9%    9.000000e+00
max      3.110000e+02
dtype: float64

In [41]:
same_moment = reviews_clean.groupby(["user_id", "timestamp"]).size()
same_moment.value_counts().sort_index()

1    2475683
2          6
Name: count, dtype: int64

In [42]:
top_users = per_user.nlargest(10).index

(
    reviews_clean[reviews_clean["user_id"].isin(top_users)]
    .groupby("user_id")["timestamp"]
    .agg(["count", "min", "max"])
    .sort_values("count", ascending=False)
)

,count,min,max
user_id,,,
AGZZXSMMS4WRHHJRBUJZI4FZDHKQ,311,2018-10-05 20:32:08.369,2023-02-27 22:43:29.344
AG73BVBKUOH22USSFJA5ZWL7AKXA,210,2015-03-08 00:17:09.000,2021-11-28 22:11:33.465
AEYVPPWR4CIKWX4BGYKCBCDL2CZQ,176,2016-11-02 13:54:46.000,2022-08-15 17:20:49.519
AFQQQ5LGNSQUEBGDCYBAZZE5T3DA,145,2019-07-21 18:01:01.189,2021-04-02 23:03:56.784
AG3HKK7MIKRQ7DOTNDOE63X2WJGQ,145,2019-08-13 16:50:48.242,2023-01-21 17:23:46.870
AH5HSMUMRJRDWBPFDQ6G3EAAOSFQ,116,2018-08-16 18:20:30.957,2023-03-15 17:17:33.490
AH665SQ6SQF6DXAGYIQFCX76LALA,112,2015-11-17 20:43:11.000,2023-03-14 07:48:02.300
AEGYU6OM2X66SMZN6AF3TSQ3N5PQ,110,2017-06-28 02:04:33.998,2023-02-08 03:42:10.424
AEULD6UDDD7XSH2HXR2L6TNRHSNQ,108,2017-12-09 00:50:35.345,2023-03-11 01:58:03.373


In [43]:
expect(
    "боты", "не найдены", True, "нет пачек в один момент; самые активные пишут годами"
)
pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,
6,повторы после удаления дублей,0.05%,True,"оставляем, могут быть разные размеры/цвета"
7,parent_asin в meta уникален,0,True,каждый товар один раз в каталоге
8,товар отзыва есть в meta,100.00%,True,
9,боты,не найдены,True,нет пачек в один момент; самые активные пишут ...


боты: у половины юзеров 1 отзыв, у 99.9% не больше 9. максимум 311, но у самых активных отзывы растянуты на несколько лет, это похоже на частых покупателей, а не на накрутку. отзывов, выпущенных пачкой в одну миллисекунду, почти нет (6 пар на 2.5 млн). ничего не удаляем

### Короткие тексты 

In [44]:
texts = reviews_ds.to_table(
    columns=["user_id", "parent_asin", "timestamp", "text"]
).to_pandas()
texts = texts.drop_duplicates(subset=["user_id", "parent_asin", "timestamp"])

n_words = texts["text"].str.split().str.len()
short_share = (n_words.fillna(0) < 3).mean()
expect("пустые / короткие (< 3 слов) тексты", f"{short_share:.2%}", short_share < 0.05)
pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,
6,повторы после удаления дублей,0.05%,True,"оставляем, могут быть разные размеры/цвета"
7,parent_asin в meta уникален,0,True,каждый товар один раз в каталоге
8,товар отзыва есть в meta,100.00%,True,
9,боты,не найдены,True,нет пачек в один момент; самые активные пишут ...


In [46]:
texts.loc[n_words < 3, "text"].sample(15, random_state=42)

1105625    great quality.
1546901         Awesome!!
1364718        Excellent!
618349          Beautiful
845040           loved it
1857786              Like
1076012          Perfect!
405433            love it
1840270     Winter jacket
40376          Super cute
1251431     Great quality
1461497        Fit great!
1750797        It’s cheap
2088973                  
401019           Good kit
Name: text, dtype: str

In [47]:
n_empty = int((n_words.fillna(0) == 0).sum())
print(n_empty, f"{n_empty / len(texts):.2%}")

2393 0.10%


In [48]:
expect(
    "решение по коротким текстам",
    "оставляем",
    True,
    "короткие отзывы осмысленные, взаимодействие настоящее",
)
del texts
pd.DataFrame(results)

,проверка,факт,ok,комментарий
0,пропуски в ключевых колонках,0,True,
1,rating только 1–5,0,True,
2,timestamp совпадает с year,0,True,
3,helpful_vote ≥ 0,0,True,
4,полные дубли строк,25244,False,сравнивали без title и text
5,повторные отзывы юзера на товар,1.06%,False,
6,повторы после удаления дублей,0.05%,True,"оставляем, могут быть разные размеры/цвета"
7,parent_asin в meta уникален,0,True,каждый товар один раз в каталоге
8,товар отзыва есть в meta,100.00%,True,
9,боты,не найдены,True,нет пачек в один момент; самые активные пишут ...


отзывов короче трёх слов 8.24%, порог < 5% не выполнен. но почти все они
осмысленные: про размер, посадку и качество ("Fit great!", "Ridiculously small"),
для одежды это нормально. совсем пустых текстов всего 0.10% (2 393 отзыва)

отзывы не удаляем: даже с коротким текстом это настоящий отзыв

## Итог части А

данные Fashion в хорошем состоянии: пропусков нет, значения корректны,
ботов не нашли, у всех отзывов есть карточка товара в каталоге

что делаем в ETL: удаляем 25 244 полных дубля (~1%).
остальное оставляем как есть. после чистки 2 475 695 отзывов